# Problem Statement


## **Business Context**


"Visit with Us," a leading travel company, is revolutionizing the tourism industry by leveraging data-driven strategies to optimize operations and customer engagement. While introducing a new package offering, such as the Wellness Tourism Package, the company faces challenges in targeting the right customers efficiently. The manual approach to identifying potential customers is inconsistent, time-consuming, and prone to errors, leading to missed opportunities and suboptimal campaign performance.



To address these issues, the company aims to implement a scalable and automated system that integrates customer data, predicts potential buyers, and enhances decision-making for marketing strategies. By utilizing an MLOps pipeline, the company seeks to achieve seamless integration of data preprocessing, model development, deployment, and CI/CD practices for continuous improvement. This system will ensure efficient targeting of customers, timely updates to the predictive model, and adaptation to evolving customer behaviors, ultimately driving growth and customer satisfaction.



## **Objective**


As an MLOps Engineer at "Visit with Us," your responsibility is to design and deploy an MLOps pipeline on GitHub to automate the end-to-end workflow for predicting customer purchases. The primary objective is to build a model that predicts whether a customer will purchase the newly introduced Wellness Tourism Package before contacting them. The pipeline will include data cleaning, preprocessing, transformation, model building, training, evaluation, and deployment, ensuring consistent performance and scalability. By leveraging GitHub Actions for CI/CD integration, the system will enable automated updates, streamline model deployment, and improve operational efficiency. This robust predictive solution will empower policymakers to make data-driven decisions, enhance marketing strategies, and effectively target potential customers, thereby driving customer acquisition and business growth.


## **Data Description**


The dataset contains customer and interaction data that serve as key attributes for predicting the likelihood of purchasing the Wellness Tourism Package. The detailed attributes are:



**Customer Details**

- **CustomerID:** Unique identifier for each customer.

- **ProdTaken:** Target variable indicating whether the customer has purchased a package (0: No, 1: Yes).

- **Age:** Age of the customer.

- **TypeofContact:** The method by which the customer was contacted (Company Invited or Self Inquiry).

- **CityTier:** The city category based on development, population, and living standards (Tier 1 > Tier 2 > Tier 3).

- **Occupation:** Customer's occupation (e.g., Salaried, Freelancer).

- **Gender:** Gender of the customer (Male, Female).

- **NumberOfPersonVisiting:** Total number of people accompanying the customer on the trip.

- **PreferredPropertyStar:** Preferred hotel rating by the customer.

- **MaritalStatus:** Marital status of the customer (Single, Married, Divorced).

- **NumberOfTrips:** Average number of trips the customer takes annually.

- **Passport:** Whether the customer holds a valid passport (0: No, 1: Yes).

- **OwnCar:** Whether the customer owns a car (0: No, 1: Yes).

- **NumberOfChildrenVisiting:** Number of children below age 5 accompanying the customer.

- **Designation:** Customer's designation in their current organization.

- **MonthlyIncome:** Gross monthly income of the customer.



**Customer Interaction Data**

- **PitchSatisfactionScore:** Score indicating the customer's satisfaction with the sales pitch.

- **ProductPitched:** The type of product pitched to the customer.

- **NumberOfFollowups:** Total number of follow-ups by the salesperson after the sales pitch.

- **DurationOfPitch:** Duration of the sales pitch delivered to the customer.



In [1]:
# Install Python dependencies for local / notebook execution.
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "datasets", "pandas", "scikit-learn", "xgboost", "mlflow", "joblib", "streamlit"]
)
print("Dependencies installed.")


Dependencies installed.


In [2]:
"""
Configure Hugging Face credentials for the cells below.

``HF_USER`` is set to your Hub username below; change the string if you use another account.
"""
import os
from getpass import getpass

# Hugging Face username (must match the account that owns the dataset/model/space).
os.environ["HF_USER"] = "snarendrababu41"

# If you see SSLCertVerificationError to huggingface.co (VPN / corporate TLS inspection),
# this disables TLS verify for Hub clients in this session (insecure). Prefer fixing
# SSL_CERT_FILE or REQUESTS_CA_BUNDLE with your organization's CA; set to "0" to enforce verify.
os.environ["HF_HUB_DISABLE_SSL_VERIFY"] = os.environ.get("HF_HUB_DISABLE_SSL_VERIFY", "1")

# Token: prefer existing env var; otherwise prompt once per session.
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("Enter HF_TOKEN (Hugging Face write token): ")

print("HF_USER=", os.environ["HF_USER"])
print("HF_TOKEN is", "set" if os.environ.get("HF_TOKEN") else "missing")


HF_USER= snarendrababu41
HF_TOKEN is set


# Model Building


In [3]:
"""
Create master project folders (master folder + data + model_building + deployment + hosting).
"""
import os
import shutil
from pathlib import Path

os.makedirs("tourism_project", exist_ok=True)
os.makedirs("tourism_project/data", exist_ok=True)
os.makedirs("tourism_project/model_building", exist_ok=True)
os.makedirs("tourism_project/deployment", exist_ok=True)
os.makedirs("tourism_project/deployment/src", exist_ok=True)
os.makedirs("tourism_project/hosting", exist_ok=True)

# Copy raw CSV from repo root into data/ if present (rubric: data under master/data).
root_csv = Path("tourism.csv")
dest = Path("tourism_project/data/tourism.csv")
if root_csv.is_file():
    shutil.copy2(root_csv, dest)
    print(f"Copied {root_csv} -> {dest}")
elif dest.is_file():
    print(f"Using existing {dest}")
else:
    print("Warning: tourism.csv not found at repo root or tourism_project/data/. Add it before data registration.")


Copied tourism.csv -> tourism_project/data/tourism.csv


## Data Registration


In [4]:
%%writefile tourism_project/hf_http_config.py
"""
Optional Hugging Face Hub HTTP/TLS settings for environments where default verification fails.

Typical cause: corporate VPN or TLS inspection presents a self-signed intermediate; Python
then raises ``SSLCertVerificationError``. Prefer fixing trust (``SSL_CERT_FILE`` or
``REQUESTS_CA_BUNDLE`` pointing at your organization's CA bundle). This module supports an
explicit opt-in to skip verification for local development only.

Parameters
----------
HF_HUB_DISABLE_SSL_VERIFY : str, optional
    When ``1``, ``true``, ``yes``, or ``on`` (case-insensitive), Hub HTTP calls in this
    process use a ``requests.Session`` with ``verify=False``. **Insecure** (traffic can be
    intercepted); do not use in production or on shared machines unless policy allows.
"""

from __future__ import annotations

import os


def _ssl_verify_disabled() -> bool:
    """
    Return whether ``HF_HUB_DISABLE_SSL_VERIFY`` requests disabling TLS verification.

    Parameters
    ----------
    None
        Reads only the process environment.
    """
    flag = os.environ.get("HF_HUB_DISABLE_SSL_VERIFY", "").strip().lower()
    return flag in ("1", "true", "yes", "on")


def apply_hf_http_settings() -> None:
    """
    Register a custom Hugging Face Hub HTTP backend when SSL verify should be skipped.

    No-op unless ``HF_HUB_DISABLE_SSL_VERIFY`` is truthy. Must run before any Hub API
    calls in the process (call at startup of CLI scripts and apps).

    Parameters
    ----------
    None
        Controlled by ``HF_HUB_DISABLE_SSL_VERIFY`` in the environment.
    """
    if not _ssl_verify_disabled():
        return

    import urllib3
    import requests
    from huggingface_hub import configure_http_backend

    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

    def backend_factory() -> requests.Session:
        """Build a session with TLS verification disabled for Hub requests."""
        session = requests.Session()
        session.verify = False
        return session

    configure_http_backend(backend_factory=backend_factory)


Overwriting tourism_project/hf_http_config.py


In [5]:
%%writefile tourism_project/model_building/data_register.py
"""
Upload raw files under ``tourism_project/data`` to a public Hugging Face Dataset.

Creates the dataset repository when it does not exist, then syncs the local data
folder to that repo.

Parameters
----------
HF_TOKEN : str
    Hugging Face API token with write access (required).
HF_USER : str, optional
    Hub username; target repo becomes ``{HF_USER}/wellness-tourism-purchase``.
HF_DATASET_REPO : str, optional
    Full dataset id ``user/repo``; overrides the default built from ``HF_USER``.
HF_HUB_DISABLE_SSL_VERIFY : str, optional
    See ``tourism_project/hf_http_config.py``; opt-in to skip TLS verify for Hub calls.
"""

from __future__ import annotations

import os
import sys
from pathlib import Path

_tp = Path(__file__).resolve().parents[1]
if str(_tp) not in sys.path:
    sys.path.insert(0, str(_tp))
import hf_http_config

hf_http_config.apply_hf_http_settings()

from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data"


def _dataset_repo_id() -> str:
    """Resolve the dataset repository id from ``HF_DATASET_REPO`` or ``HF_USER``."""
    explicit = os.getenv("HF_DATASET_REPO")
    if explicit:
        return explicit.strip()
    user = os.getenv("HF_USER", "").strip()
    if not user:
        raise ValueError(
            "Set HF_USER (username) or HF_DATASET_REPO (full dataset repo id)."
        )
    return f"{user}/wellness-tourism-purchase"


def main() -> None:
    """Ensure the Hub dataset exists and upload ``tourism_project/data`` to it."""
    token = os.getenv("HF_TOKEN")
    if not token:
        raise ValueError("HF_TOKEN environment variable is required for uploads.")

    repo_id = _dataset_repo_id()
    repo_type = "dataset"
    api = HfApi(token=token)

    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

    try:
        api.repo_info(repo_id=repo_id, repo_type=repo_type)
        print(f"Dataset repo '{repo_id}' already exists. Uploading files.")
    except RepositoryNotFoundError:
        print(f"Dataset repo '{repo_id}' not found. Creating public repo...")
        create_repo(repo_id=repo_id, repo_type=repo_type, private=False, token=token)
        print(f"Dataset repo '{repo_id}' created.")

    api.upload_folder(
        folder_path=str(DATA_DIR),
        repo_id=repo_id,
        repo_type=repo_type,
        token=token,
    )
    print(f"Uploaded contents of {DATA_DIR} to {repo_id}.")


if __name__ == "__main__":
    main()


Overwriting tourism_project/model_building/data_register.py


In [6]:
"""Run dataset registration (upload raw files to the Hub)."""
import subprocess
import sys

subprocess.check_call([sys.executable, "tourism_project/model_building/data_register.py"])
print("Data registration complete.")


Dataset repo 'snarendrababu41/wellness-tourism-purchase' already exists. Uploading files.


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded contents of /Users/narendrababu.sinappa/Desktop/mlops/tourism_project/data to snarendrababu41/wellness-tourism-purchase.
Data registration complete.


### Observations — data registration & versioning

- The Hub holds one **versioned** copy of `tourism.csv` for analytics, training, and CI.
- Every upload is **traceable**, so you can tie a model back to the data snapshot it saw.
- Run the **same** `data_register.py` locally and in GitHub Actions to avoid silent drift.


## Data Preparation


Loads `tourism.csv` from the Hub via `hf://datasets/...`, cleans features, performs an 80/20 stratified split, saves `Xtrain.csv`, `Xtest.csv`, `ytrain.csv`, `ytest.csv` under `tourism_project/data/`, and uploads them to the same dataset repo.



In [7]:
%%writefile tourism_project/model_building/prep.py
"""
Load ``tourism.csv`` from the Hub, clean features, split train/test, save CSVs locally,
and upload the split files back to the same Dataset repository.

Parameters
----------
HF_TOKEN : str
    Required for uploading split files to the Hub.
HF_USER : str, optional
    Hub username for default dataset id (see ``data_register``).
HF_DATASET_REPO : str, optional
    Full dataset id; overrides default built from ``HF_USER``.
"""

from __future__ import annotations

import os
import sys
from pathlib import Path

_tp = Path(__file__).resolve().parents[1]
if str(_tp) not in sys.path:
    sys.path.insert(0, str(_tp))
import hf_http_config

hf_http_config.apply_hf_http_settings()

import pandas as pd
from huggingface_hub import HfApi
from sklearn.model_selection import train_test_split

ROOT = Path(__file__).resolve().parents[1]
DATA_DIR = ROOT / "data"

TARGET_COL = "ProdTaken"
DROP_COLS = {"CustomerID"}

NUMERIC_FEATURES = [
    "Age",
    "CityTier",
    "DurationOfPitch",
    "NumberOfPersonVisiting",
    "NumberOfFollowups",
    "PreferredPropertyStar",
    "NumberOfTrips",
    "Passport",
    "PitchSatisfactionScore",
    "OwnCar",
    "NumberOfChildrenVisiting",
    "MonthlyIncome",
]
CATEGORICAL_FEATURES = [
    "TypeofContact",
    "Occupation",
    "Gender",
    "ProductPitched",
    "MaritalStatus",
    "Designation",
]


def _dataset_repo_id() -> str:
    """Resolve the dataset repository id from ``HF_DATASET_REPO`` or ``HF_USER``."""
    explicit = os.getenv("HF_DATASET_REPO")
    if explicit:
        return explicit.strip()
    user = os.getenv("HF_USER", "").strip()
    if not user:
        raise ValueError(
            "Set HF_USER (username) or HF_DATASET_REPO (full dataset repo id)."
        )
    return f"{user}/wellness-tourism-purchase"


def _hf_csv_uri(filename: str) -> str:
    """
    Build an ``hf://datasets/...`` URI for a file in the registered dataset.

    Parameters
    ----------
    filename : str
        File name inside the dataset repo (e.g. ``tourism.csv``).
    """
    return f"hf://datasets/{_dataset_repo_id()}/{filename}"


def load_raw_from_hub() -> pd.DataFrame:
    """Read ``tourism.csv`` from the Hub into a DataFrame."""
    path = _hf_csv_uri("tourism.csv")
    df = pd.read_csv(path)
    print(f"Loaded dataset from {path} shape={df.shape}.")
    return df


def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop ID columns, impute missing values, and keep modeling columns plus target.

    Parameters
    ----------
    df : pd.DataFrame
        Raw tourism table from ``tourism.csv``.
    """
    out = df.copy()
    unnamed = [c for c in out.columns if c.startswith("Unnamed")]
    if unnamed:
        out = out.drop(columns=unnamed)
    out = out.drop(columns=[c for c in DROP_COLS if c in out.columns], errors="ignore")

    feature_cols = [c for c in NUMERIC_FEATURES + CATEGORICAL_FEATURES if c in out.columns]
    missing = set(NUMERIC_FEATURES + CATEGORICAL_FEATURES + [TARGET_COL]) - set(
        out.columns
    )
    if missing:
        raise ValueError(f"Dataset missing expected columns: {sorted(missing)}")

    for col in NUMERIC_FEATURES:
        med = out[col].median()
        out[col] = out[col].fillna(med)
    for col in CATEGORICAL_FEATURES:
        out[col] = out[col].fillna("Unknown").astype(str)

    out["Gender"] = out["Gender"].replace({"Fe Male": "Female"})

    out = out[feature_cols + [TARGET_COL]].dropna(subset=[TARGET_COL])
    out[TARGET_COL] = out[TARGET_COL].astype(int)
    return out


def main() -> None:
    """Run load, clean, stratified split, local save, and Hub upload of split CSVs."""
    token = os.getenv("HF_TOKEN")
    if not token:
        raise ValueError("HF_TOKEN environment variable is required for uploads.")

    DATA_DIR.mkdir(parents=True, exist_ok=True)

    raw = load_raw_from_hub()
    clean = clean_dataframe(raw)
    print(f"After cleaning: shape={clean.shape}")

    y = clean[TARGET_COL]
    X = clean.drop(columns=[TARGET_COL])

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y,
    )

    paths = {
        "Xtrain.csv": X_train,
        "Xtest.csv": X_test,
        "ytrain.csv": y_train,
        "ytest.csv": y_test,
    }
    for name, frame in paths.items():
        fp = DATA_DIR / name
        frame.to_csv(fp, index=False)
        print(f"Wrote {fp}")

    repo_id = _dataset_repo_id()
    api = HfApi(token=token)
    for name in paths:
        api.upload_file(
            path_or_fileobj=str(DATA_DIR / name),
            path_in_repo=name,
            repo_id=repo_id,
            repo_type="dataset",
            token=token,
        )
        print(f"Uploaded {name} to {repo_id}.")


if __name__ == "__main__":
    main()


Overwriting tourism_project/model_building/prep.py


In [8]:
"""Execute data preparation pipeline."""
import subprocess
import sys

subprocess.check_call([sys.executable, "tourism_project/model_building/prep.py"])
print("Data preparation complete.")


Loaded dataset from hf://datasets/snarendrababu41/wellness-tourism-purchase/tourism.csv shape=(4128, 21).
After cleaning: shape=(4128, 19)
Wrote /Users/narendrababu.sinappa/Desktop/mlops/tourism_project/data/Xtrain.csv
Wrote /Users/narendrababu.sinappa/Desktop/mlops/tourism_project/data/Xtest.csv
Wrote /Users/narendrababu.sinappa/Desktop/mlops/tourism_project/data/ytrain.csv
Wrote /Users/narendrababu.sinappa/Desktop/mlops/tourism_project/data/ytest.csv


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded Xtrain.csv to snarendrababu41/wellness-tourism-purchase.


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded Xtest.csv to snarendrababu41/wellness-tourism-purchase.


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded ytrain.csv to snarendrababu41/wellness-tourism-purchase.


No files have been modified since last commit. Skipping to prevent empty commit.


Uploaded ytest.csv to snarendrababu41/wellness-tourism-purchase.
Data preparation complete.


### Observations — preparation & train/test design

- **Stratified 80/20** keeps a similar buyer rate in train and test for realistic metrics.
- **Imputation** (medians / `'Unknown'`) keeps every row scorable in training and in Streamlit.


## Model Training and Registration with Experimentation Tracking


Uses **XGBoost** inside a sklearn **pipeline** (scaling + one-hot encoding). **GridSearchCV** tunes hyperparameters; every combination is logged to **MLflow** (file store under `tourism_project/mlruns`). The best model is saved as `best_wellness_tourism_model.joblib` and uploaded to the **Hugging Face Model Hub**.



In [9]:
%%writefile tourism_project/model_building/train.py
"""
Tune an XGBoost pipeline on Hub-hosted train/test CSVs, log runs to MLflow, and
upload the best ``joblib`` model to the Hugging Face Model Hub.

Parameters
----------
HF_TOKEN : str
    Required for creating the model repo and uploading the artifact.
HF_USER : str, optional
    Username for default dataset and model repo ids when overrides are not set.
HF_DATASET_REPO : str, optional
    Dataset id containing ``Xtrain.csv``, ``Xtest.csv``, ``ytrain.csv``, ``ytest.csv``.
HF_MODEL_REPO : str, optional
    Model id for the uploaded ``joblib`` file; default ``{HF_USER}/wellness-tourism-xgboost-model``.

Notes
-----
On **macOS**, the pip XGBoost wheel needs OpenMP (``libomp.dylib``). Run ``brew install libomp``.
The formula is *keg-only*, so XGBoost often cannot find the library unless you also run
``brew link libomp --force`` (or start Python with ``DYLD_LIBRARY_PATH`` set to
``$(brew --prefix)/opt/libomp/lib``). Linux CI images typically already ship compatible OpenMP.
"""

from __future__ import annotations

import os
import sys
from pathlib import Path

_tp = Path(__file__).resolve().parents[1]
if str(_tp) not in sys.path:
    sys.path.insert(0, str(_tp))
import hf_http_config

hf_http_config.apply_hf_http_settings()

import joblib
import mlflow
import numpy as np
import pandas as pd
import xgboost as xgb
from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError
from sklearn.compose import make_column_transformer
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC_FEATURES = [
    "Age",
    "CityTier",
    "DurationOfPitch",
    "NumberOfPersonVisiting",
    "NumberOfFollowups",
    "PreferredPropertyStar",
    "NumberOfTrips",
    "Passport",
    "PitchSatisfactionScore",
    "OwnCar",
    "NumberOfChildrenVisiting",
    "MonthlyIncome",
]
CATEGORICAL_FEATURES = [
    "TypeofContact",
    "Occupation",
    "Gender",
    "ProductPitched",
    "MaritalStatus",
    "Designation",
]

ROOT = Path(__file__).resolve().parents[1]
MLRUNS_DIR = ROOT / "mlruns"
MODEL_FILENAME = "best_wellness_tourism_model.joblib"


def _dataset_repo_id() -> str:
    """Resolve the dataset repository id from ``HF_DATASET_REPO`` or ``HF_USER``."""
    explicit = os.getenv("HF_DATASET_REPO")
    if explicit:
        return explicit.strip()
    user = os.getenv("HF_USER", "").strip()
    if not user:
        raise ValueError(
            "Set HF_USER (username) or HF_DATASET_REPO (full dataset repo id)."
        )
    return f"{user}/wellness-tourism-purchase"


def _model_repo_id() -> str:
    """Resolve the model repository id from ``HF_MODEL_REPO`` or ``HF_USER``."""
    explicit = os.getenv("HF_MODEL_REPO")
    if explicit:
        return explicit.strip()
    user = os.getenv("HF_USER", "").strip()
    if not user:
        raise ValueError(
            "Set HF_USER (username) or HF_MODEL_REPO (full model repo id)."
        )
    return f"{user}/wellness-tourism-xgboost-model"


def _load_xy() -> tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray]:
    """Load train/test feature matrices and labels from the Hub via ``hf://`` URIs."""
    base = _dataset_repo_id()
    X_train = pd.read_csv(f"hf://datasets/{base}/Xtrain.csv")
    X_test = pd.read_csv(f"hf://datasets/{base}/Xtest.csv")
    y_train = pd.read_csv(f"hf://datasets/{base}/ytrain.csv").squeeze("columns")
    y_test = pd.read_csv(f"hf://datasets/{base}/ytest.csv").squeeze("columns")
    y_train = np.asarray(y_train).ravel()
    y_test = np.asarray(y_test).ravel()
    return X_train, X_test, y_train, y_test


def main() -> None:
    """Run grid search, MLflow logging, evaluation, ``joblib`` export, and Hub upload."""
    token = os.getenv("HF_TOKEN")
    if not token:
        raise ValueError("HF_TOKEN environment variable is required.")

    MLRUNS_DIR.mkdir(parents=True, exist_ok=True)
    mlflow.set_tracking_uri(f"file:{MLRUNS_DIR}")
    mlflow.set_experiment("wellness-tourism-xgboost")

    X_train, X_test, y_train, y_test = _load_xy()
    print(f"Train shape={X_train.shape}, Test shape={X_test.shape}")

    pos = (y_train == 1).sum()
    neg = (y_train == 0).sum()
    scale_pos_weight = float(neg) / float(pos) if pos > 0 else 1.0

    preprocessor = make_column_transformer(
        (StandardScaler(), NUMERIC_FEATURES),
        (OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    )
    xgb_model = xgb.XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric="logloss",
    )
    model_pipeline = make_pipeline(preprocessor, xgb_model)

    param_grid = {
        "xgbclassifier__n_estimators": [100],
        "xgbclassifier__max_depth": [3, 5],
        "xgbclassifier__learning_rate": [0.05, 0.1],
        "xgbclassifier__colsample_bytree": [0.8],
        "xgbclassifier__reg_lambda": [0.5, 1.0],
    }

    api = HfApi(token=token)
    model_repo = _model_repo_id()

    with mlflow.start_run(run_name="wellness_tourism_grid_search"):
        grid_search = GridSearchCV(
            model_pipeline,
            param_grid,
            cv=5,
            n_jobs=-1,
            scoring="f1",
        )
        grid_search.fit(X_train, y_train)

        results = grid_search.cv_results_
        for i in range(len(results["params"])):
            with mlflow.start_run(nested=True):
                mlflow.log_params(results["params"][i])
                mlflow.log_metric("mean_test_score", float(results["mean_test_score"][i]))
                mlflow.log_metric("std_test_score", float(results["std_test_score"][i]))

        mlflow.log_params(grid_search.best_params_)

        best_model = grid_search.best_estimator_
        threshold = 0.5

        y_pred_train = (best_model.predict_proba(X_train)[:, 1] >= threshold).astype(int)
        y_pred_test = (best_model.predict_proba(X_test)[:, 1] >= threshold).astype(int)

        train_report = classification_report(
            y_train, y_pred_train, output_dict=True, zero_division=0
        )
        test_report = classification_report(
            y_test, y_pred_test, output_dict=True, zero_division=0
        )

        mlflow.log_metrics(
            {
                "train_accuracy": train_report["accuracy"],
                "train_precision_class1": train_report["1"]["precision"],
                "train_recall_class1": train_report["1"]["recall"],
                "train_f1_class1": train_report["1"]["f1-score"],
                "test_accuracy": test_report["accuracy"],
                "test_precision_class1": test_report["1"]["precision"],
                "test_recall_class1": test_report["1"]["recall"],
                "test_f1_class1": test_report["1"]["f1-score"],
            }
        )

        print("Best params:", grid_search.best_params_)
        print("Train report:\n", classification_report(y_train, y_pred_train))
        print("Test report:\n", classification_report(y_test, y_pred_test))

        out_path = Path(MODEL_FILENAME)
        joblib.dump(best_model, out_path)
        mlflow.log_artifact(str(out_path), artifact_path="model")

        try:
            api.repo_info(repo_id=model_repo, repo_type="model")
            print(f"Model repo '{model_repo}' exists.")
        except RepositoryNotFoundError:
            print(f"Creating public model repo '{model_repo}'...")
            create_repo(repo_id=model_repo, repo_type="model", private=False, token=token)

        api.upload_file(
            path_or_fileobj=str(out_path),
            path_in_repo=MODEL_FILENAME,
            repo_id=model_repo,
            repo_type="model",
            token=token,
        )
        print(f"Uploaded {MODEL_FILENAME} to {model_repo}.")


if __name__ == "__main__":
    main()


Overwriting tourism_project/model_building/train.py


In [10]:
"""Train, tune, log experiments, evaluate, and push the best model to the Hub."""
import os
import shutil
import subprocess
import sys
from pathlib import Path


def _darwin_libomp_library_dir() -> str | None:
    """
    Return the directory that contains Homebrew's ``libomp.dylib`` on macOS.

    Parameters
    ----------
    None
        Uses ``HOMEBREW_PREFIX``, ``brew`` on ``PATH``, ``/opt/homebrew``, and ``/usr/local``.
    """
    if sys.platform != "darwin":
        return None
    candidates: list[Path] = []
    hp = os.environ.get("HOMEBREW_PREFIX", "").strip()
    if hp:
        candidates.append(Path(hp) / "opt" / "libomp" / "lib")
    brew = shutil.which("brew")
    if brew:
        candidates.append(Path(brew).resolve().parent.parent / "opt" / "libomp" / "lib")
    candidates.extend(
        [Path("/opt/homebrew/opt/libomp/lib"), Path("/usr/local/opt/libomp/lib")]
    )
    for d in candidates:
        if (d / "libomp.dylib").is_file():
            return str(d)
    return None


_train_env = os.environ.copy()
_libomp_dir = _darwin_libomp_library_dir()
if _libomp_dir:
    _prev = _train_env.get("DYLD_LIBRARY_PATH", "")
    _train_env["DYLD_LIBRARY_PATH"] = _libomp_dir + (":" + _prev if _prev else "")

subprocess.check_call(
    [sys.executable, "tourism_project/model_building/train.py"],
    env=_train_env,
)
print("Model training and Hub upload complete.")


Train shape=(3302, 18), Test shape=(826, 18)
Best params: {'xgbclassifier__colsample_bytree': 0.8, 'xgbclassifier__learning_rate': 0.1, 'xgbclassifier__max_depth': 5, 'xgbclassifier__n_estimators': 100, 'xgbclassifier__reg_lambda': 1.0}
Train report:
               precision    recall  f1-score   support

           0       0.99      0.95      0.97      2664
           1       0.82      0.95      0.88       638

    accuracy                           0.95      3302
   macro avg       0.90      0.95      0.92      3302
weighted avg       0.95      0.95      0.95      3302

Test report:
               precision    recall  f1-score   support

           0       0.94      0.91      0.92       667
           1       0.66      0.77      0.71       159

    accuracy                           0.88       826
   macro avg       0.80      0.84      0.82       826
weighted avg       0.89      0.88      0.88       826

Model repo 'snarendrababu41/wellness-tourism-xgboost-model' exists.
Uploaded bes

### Observations — modelling, tracking, and model registry

- **XGBoost** inside one sklearn **pipeline** packages preprocessing + model for Hub and Space.
- **MLflow** stores each grid point; **`best_wellness_tourism_model.joblib`** on the Hub is the single artifact to serve.


# Deployment


## Dockerfile


**Dockerfile configurations** :

- **Base image:** `python:3.10-slim`
- **WORKDIR:** `/app`
- **Dependencies:** `pip install -r requirements.txt` (pinned in `deployment/requirements.txt`)
- **Artifacts copied:** `requirements.txt`, then full deployment context (`src/streamlit_app.py`, etc.)
- **EXPOSE:** `7860` (must match ``README.md`` ``app_port``)
- **ENTRYPOINT:** `streamlit run src/streamlit_app.py` with `--server.port=7860`, `--server.address=0.0.0.0`, plus Space-friendly flags; **HEALTHCHECK** curls `/_stcore/health`.



In [ ]:
%%writefile tourism_project/deployment/Dockerfile
# Hugging Face Space (Docker SDK): listen on 0.0.0.0:7860 (must match README app_port).
# Base Python 3.10 matches GitHub Actions / training (tourism_project/requirements.txt).
# - curl: HEALTHCHECK
# - libgomp1: XGBoost shared library on Debian slim
# Build logs often stay empty until each RUN layer completes (pip can take many minutes).
FROM python:3.10-slim

WORKDIR /app

RUN apt-get update && apt-get install -y --no-install-recommends \
    ca-certificates \
    curl \
    libgomp1 \
    && rm -rf /var/lib/apt/lists/*

# Verbose logs in `docker logs` / Space build & runtime logs.
ENV STREAMLIT_LOGGER_LEVEL=debug \
    HF_HUB_VERBOSITY=debug \
    STREAMLIT_SERVER_HEADLESS=true \
    STREAMLIT_BROWSER_GATHER_USAGE_STATS=false \
    PIP_NO_CACHE_DIR=1 \
    PIP_DISABLE_PIP_VERSION_CHECK=1 \
    PIP_DEFAULT_TIMEOUT=120 \
    PYTHONUNBUFFERED=1

COPY requirements.txt ./
COPY src/ ./src/

RUN echo "=== HF build: pip install start ===" \
    && pip3 install --no-cache-dir --upgrade pip \
    && pip3 install --no-cache-dir --prefer-binary -r requirements.txt \
    && echo "=== HF build: pip install finished ==="

COPY . .

EXPOSE 7860

# Requires curl (installed above). start-period allows Streamlit to boot before first check.
HEALTHCHECK --interval=30s --timeout=10s --start-period=120s --retries=5 \
    CMD curl --fail --silent http://127.0.0.1:7860/_stcore/health || exit 1

# --server.address=0.0.0.0 is required: otherwise Streamlit binds to localhost only and the Space proxy cannot reach the app.
ENTRYPOINT ["streamlit", "run", "src/streamlit_app.py", "--logger.level=debug", "--server.port=7860", "--server.address=0.0.0.0", "--server.enableXsrfProtection=false", "--server.fileWatcherType", "none", "--browser.gatherUsageStats", "false"]


In [12]:
%%writefile tourism_project/deployment/.dockerignore
# Keep Space build context tiny and avoid leaking junk into the image.
__pycache__
*.py[cod]
*.egg-info
.git
.gitignore
.env
.venv
*.md
!README.md


Overwriting tourism_project/deployment/.dockerignore


In [13]:
%%writefile tourism_project/deployment/README.md
---
title: "Wellness Tourism Purchase Predictor"
emoji: 🧳
colorFrom: blue
colorTo: indigo
sdk: docker
app_port: 7860
startup_duration_timeout: 45m
---

Front-end Streamlit app for tourism prediction. The UI lives in **`src/streamlit_app.py`**; the `Dockerfile` uses **`ENTRYPOINT`** with **`HEALTHCHECK`** on `/_stcore/health` (Python **3.10-slim** base).

Overwriting tourism_project/deployment/README.md


## Streamlit App


The Streamlit script is ``src/streamlit_app.py`` (see ``Dockerfile`` ``ENTRYPOINT``). It downloads the trained pipeline from the Model Hub and builds a ``pandas.DataFrame`` from user inputs.



In [ ]:
%%writefile tourism_project/deployment/src/streamlit_app.py
"""
Wellness Tourism purchase prediction UI: collect features, load the trained pipeline
from the Hugging Face Model Hub, and display ``predict_proba`` for class 1.

The model is loaded lazily on **Predict** (or via **Load model only** in the sidebar).
Use **Clear model cache** after changing Space secrets or uploading a new joblib.

Parameters
----------
HF_MODEL_REPO : str, optional
    Model Hub id for ``hf_hub_download``; default ``snarendrababu41/wellness-tourism-xgboost-model``.
HF_MODEL_FILENAME : str, optional
    Artifact filename in that repo; default ``best_wellness_tourism_model.joblib``.
HF_TOKEN : str, optional
    Required when the model repo is private or gated (set as a Space repository secret).
HF_HUB_DISABLE_SSL_VERIFY : str, optional
    If ``hf_http_config.py`` is present at the app root, optional TLS override for Hub calls.
"""

from __future__ import annotations

import os
import sys
from pathlib import Path

_script_dir = Path(__file__).resolve().parent
for _root in (_script_dir.parent, _script_dir):
    _cfg = _root / "hf_http_config.py"
    if _cfg.is_file():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        import hf_http_config

        hf_http_config.apply_hf_http_settings()
        break

import joblib
import pandas as pd
import streamlit as st
from huggingface_hub import hf_hub_download

DEFAULT_MODEL_REPO = "snarendrababu41/wellness-tourism-xgboost-model"
DEFAULT_MODEL_FILE = "best_wellness_tourism_model.joblib"
CLASSIFICATION_THRESHOLD = 0.5


def _model_repo() -> str:
    """Return the effective model Hub id from ``HF_MODEL_REPO`` or the built-in default."""
    return os.environ.get("HF_MODEL_REPO", DEFAULT_MODEL_REPO).strip()


def _model_filename() -> str:
    """Return the joblib filename from ``HF_MODEL_FILENAME`` or the default."""
    return os.environ.get("HF_MODEL_FILENAME", DEFAULT_MODEL_FILE).strip()


@st.cache_resource(show_spinner="Downloading model from the Hub…")
def load_model(repo_id: str, filename: str):
    """
    Download and deserialize the joblib pipeline for the given Hub repo and file.

    Parameters
    ----------
    repo_id : str
        Hugging Face model repository id (``user/repo``).
    filename : str
        Path of the artifact inside that repository.

    Returns
    -------
    object
        Deserialized sklearn pipeline (supports ``predict_proba``).
    """
    path = hf_hub_download(repo_id=repo_id, filename=filename)
    return joblib.load(path)


def _build_input_row(
    age: float,
    city_tier: int,
    duration_pitch: float,
    n_visiting: int,
    n_followups: float,
    product_pitched: str,
    pref_star: float,
    marital: str,
    n_trips: float,
    passport: int,
    pitch_score: int,
    own_car: int,
    n_children: float,
    designation: str,
    monthly_income: float,
    type_contact: str,
    occupation: str,
    gender: str,
) -> pd.DataFrame:
    """
    Assemble a single-row feature ``DataFrame`` matching training column names.

    Parameters
    ----------
    age : float
        Customer age in years.
    city_tier : int
        City tier (1–3).
    duration_pitch : float
        Pitch duration in minutes.
    n_visiting : int
        Number of persons visiting.
    n_followups : float
        Number of follow-ups.
    product_pitched : str
        Product category pitched.
    pref_star : float
        Preferred property star rating.
    marital : str
        Marital status label.
    n_trips : float
        Number of trips per year.
    passport : int
        Binary passport indicator (0/1).
    pitch_score : int
        Pitch satisfaction score (1–5).
    own_car : int
        Binary car ownership (0/1).
    n_children : float
        Children under 5 visiting.
    designation : str
        Job designation label.
    monthly_income : float
        Monthly income.
    type_contact : str
        Type of contact (e.g. Self Enquiry).
    occupation : str
        Occupation category.
    gender : str
        Gender label.

    Returns
    -------
    pandas.DataFrame
        One row with columns aligned to the training schema.
    """
    return pd.DataFrame(
        [
            {
                "Age": age,
                "CityTier": city_tier,
                "DurationOfPitch": duration_pitch,
                "NumberOfPersonVisiting": n_visiting,
                "NumberOfFollowups": n_followups,
                "PreferredPropertyStar": pref_star,
                "NumberOfTrips": n_trips,
                "Passport": passport,
                "PitchSatisfactionScore": pitch_score,
                "OwnCar": own_car,
                "NumberOfChildrenVisiting": n_children,
                "MonthlyIncome": monthly_income,
                "TypeofContact": type_contact,
                "Occupation": occupation,
                "Gender": gender,
                "ProductPitched": product_pitched,
                "MaritalStatus": marital,
                "Designation": designation,
            }
        ]
    )


def main() -> None:
    """
    Render the app: sidebar for testing helpers, form inputs, and prediction output.

    Parameters
    ----------
    None
        Reads environment variables and Streamlit widget state.
    """
    st.set_page_config(
        page_title="Wellness Tourism — Purchase prediction",
        page_icon="🧳",
        layout="wide",
    )

    repo = _model_repo()
    artifact = _model_filename()
    token_set = bool(os.environ.get("HF_TOKEN", "").strip())

    with st.sidebar:
        st.header("Testing")
        st.markdown(
            "Use this panel to confirm Hub settings and refresh the model after "
            "you change Space secrets or upload a new artifact."
        )
        st.text_input("Resolved `HF_MODEL_REPO`", value=repo, disabled=True)
        st.text_input("Resolved `HF_MODEL_FILENAME`", value=artifact, disabled=True)
        st.caption(f"`HF_TOKEN` secret: **{'set' if token_set else 'not set'}**")
        if st.button("Clear model cache", type="secondary"):
            load_model.clear()
            st.success("Cache cleared. Run **Predict** or **Load model only** again.")
        if st.button("Load model only", type="primary"):
            try:
                _ = load_model(repo, artifact)
                st.success("Model downloaded and loaded OK.")
            except Exception as exc:  # noqa: BLE001
                st.error(f"Load failed: {exc}")
        with st.expander("Checklist"):
            st.markdown(
                """
                - [ ] Space secret **`HF_MODEL_REPO`** = your `user/wellness-tourism-xgboost-model`
                - [ ] If repo is **private**: **`HF_TOKEN`** (read) is set on the Space
                - [ ] Artifact **`best_wellness_tourism_model.joblib`** exists on that repo
                - [ ] After changing secrets, click **Clear model cache**
                """
            )

    st.title("Wellness Tourism Package — Purchase prediction")
    st.write(
        "Estimate whether a customer is likely to purchase the **Wellness Tourism "
        "Package** (Visit with Us) from profile and pitch attributes."
    )

    left, right = st.columns((2, 1))
    with left:
        st.subheader("Inputs")
        c1, c2 = st.columns(2)
        with c1:
            age = st.number_input("Age", min_value=18.0, max_value=100.0, value=35.0)
            city_tier = st.selectbox("City tier", [1, 2, 3], index=0)
            duration_pitch = st.number_input(
                "Duration of pitch (minutes)", min_value=0.0, value=10.0
            )
            n_visiting = st.number_input(
                "Number of persons visiting", min_value=1, value=2
            )
            n_followups = st.number_input(
                "Number of follow-ups", min_value=0.0, value=3.0
            )
            product_pitched = st.selectbox(
                "Product pitched",
                ["Basic", "Deluxe", "Standard", "Super Deluxe", "King"],
            )
            pref_star = st.number_input(
                "Preferred property star", min_value=1.0, max_value=5.0, value=3.0
            )
            marital = st.selectbox(
                "Marital status",
                ["Single", "Married", "Divorced", "Unmarried"],
            )
            n_trips = st.number_input(
                "Number of trips per year", min_value=0.0, value=2.0
            )
        with c2:
            passport = st.selectbox(
                "Passport", [0, 1], format_func=lambda x: "Yes" if x else "No"
            )
            pitch_score = st.slider("Pitch satisfaction score", 1, 5, 3)
            own_car = st.selectbox(
                "Owns car", [0, 1], format_func=lambda x: "Yes" if x else "No"
            )
            n_children = st.number_input(
                "Children under 5 visiting", min_value=0.0, value=0.0
            )
            designation = st.selectbox(
                "Designation",
                ["Executive", "Manager", "Senior Manager", "AVP", "VP"],
            )
            monthly_income = st.number_input(
                "Monthly income", min_value=0.0, value=20000.0
            )
            type_contact = st.selectbox(
                "Type of contact",
                ["Self Enquiry", "Company Invited"],
            )
            occupation = st.selectbox(
                "Occupation",
                ["Salaried", "Free Lancer", "Small Business", "Large Business"],
            )
            gender = st.selectbox("Gender", ["Male", "Female"])

        input_row = _build_input_row(
            age,
            city_tier,
            duration_pitch,
            n_visiting,
            n_followups,
            product_pitched,
            pref_star,
            marital,
            n_trips,
            passport,
            pitch_score,
            own_car,
            n_children,
            designation,
            monthly_income,
            type_contact,
            occupation,
            gender,
        )

        predict = st.button("Predict", type="primary")

    with right:
        st.subheader("Result")
        if predict:
            try:
                with st.spinner("Loading model and scoring…"):
                    model = load_model(repo, artifact)
                    proba = float(model.predict_proba(input_row)[0, 1])
            except Exception as exc:  # noqa: BLE001
                st.error("Prediction failed.")
                st.code(str(exc), language="text")
                with st.expander("Full traceback (for debugging)"):
                    st.exception(exc)
                st.info(
                    "Verify Space secrets **`HF_MODEL_REPO`** / **`HF_TOKEN`**, "
                    "then **Clear model cache** and retry."
                )
            else:
                pred = int(proba >= CLASSIFICATION_THRESHOLD)
                label = "Likely buyer (1)" if pred else "Unlikely buyer (0)"
                st.metric("P(purchase)", f"{proba:.2%}")
                st.success(
                    f"Predicted class: **{label}** (threshold {CLASSIFICATION_THRESHOLD})."
                )
        else:
            st.caption("Adjust inputs and click **Predict**.")

        with st.expander("Feature row (debug)"):
            st.dataframe(input_row, use_container_width=True)


if __name__ == "__main__":
    main()


## Dependency Handling


Deployment dependencies (pinned) for the Docker Space:



In [ ]:
%%writefile tourism_project/deployment/requirements.txt
# Pinned for Python 3.10.x (matches GitHub Actions & training; see Dockerfile FROM).
# Versions align with tourism_project/requirements.txt for joblib/sklearn/xgboost parity.
numpy==2.2.2
pandas==2.2.2
huggingface_hub==0.32.6
streamlit==1.45.0
joblib==1.5.1
scikit-learn==1.6.0
xgboost==2.1.4


# Hosting


Script uploads all files under `tourism_project/deployment/` to the public Space `wellness-tourism-streamlit`.



In [16]:
%%writefile tourism_project/hosting/hosting.py
"""
Upload ``tourism_project/deployment`` to a Hugging Face Space (Docker SDK) so the
Streamlit image can build from the Dockerfile.

Parameters
----------
HF_TOKEN : str
    Token with write access to Spaces (required).
HF_USER : str, optional
    Username; default Space id ``{HF_USER}/wellness-tourism-streamlit``.
HF_SPACE_REPO : str, optional
    Full Space id ``user/space-name``; overrides default built from ``HF_USER``.
"""

from __future__ import annotations

import os
import shutil
import sys
from pathlib import Path

_tp = Path(__file__).resolve().parents[1]
if str(_tp) not in sys.path:
    sys.path.insert(0, str(_tp))
import hf_http_config

hf_http_config.apply_hf_http_settings()

from huggingface_hub import HfApi, create_repo
from huggingface_hub.utils import RepositoryNotFoundError

ROOT = Path(__file__).resolve().parents[1]
DEPLOY_DIR = ROOT / "deployment"
HF_CFG_SRC = ROOT / "hf_http_config.py"
HF_CFG_STAGING = DEPLOY_DIR / "hf_http_config.py"


def _space_repo_id() -> str:
    """Resolve the Space repository id from ``HF_SPACE_REPO`` or ``HF_USER``."""
    explicit = os.getenv("HF_SPACE_REPO")
    if explicit:
        return explicit.strip()
    user = os.getenv("HF_USER", "").strip()
    if not user:
        raise ValueError(
            "Set HF_USER (username) or HF_SPACE_REPO (full Space repo id)."
        )
    return f"{user}/wellness-tourism-streamlit"


def main() -> None:
    """
    Create the Space if missing and upload the deployment folder in one commit.

    Stages ``tourism_project/hf_http_config.py`` into ``deployment/`` before upload so
    the Space repo root always contains ``hf_http_config.py`` alongside ``Dockerfile``
    (avoids a separate upload race and matches ``COPY . .`` in the image).

    Parameters
    ----------
    None
        Uses ``HF_TOKEN``, ``HF_USER`` or ``HF_SPACE_REPO`` from the environment.
    """
    token = os.getenv("HF_TOKEN")
    if not token:
        raise ValueError("HF_TOKEN environment variable is required.")

    if not DEPLOY_DIR.is_dir():
        raise FileNotFoundError(f"Deployment folder not found: {DEPLOY_DIR}")

    if not (DEPLOY_DIR / "src" / "streamlit_app.py").is_file():
        raise FileNotFoundError(
            f"Missing Streamlit app: {DEPLOY_DIR / 'src' / 'streamlit_app.py'}"
        )
    if not (DEPLOY_DIR / "Dockerfile").is_file():
        raise FileNotFoundError(f"Missing Dockerfile in {DEPLOY_DIR}")

    repo_id = _space_repo_id()
    api = HfApi(token=token)
    try:
        api.repo_info(repo_id=repo_id, repo_type="space")
    except RepositoryNotFoundError:
        create_repo(
            repo_id=repo_id,
            repo_type="space",
            private=False,
            token=token,
            space_sdk="docker",
        )
        print(f"Created Space {repo_id} (docker). Configure SDK in Hub UI if needed.")

    copied_cfg = False
    if HF_CFG_SRC.is_file():
        shutil.copy2(HF_CFG_SRC, HF_CFG_STAGING)
        copied_cfg = True
    try:
        api.upload_folder(
            folder_path=str(DEPLOY_DIR),
            repo_id=repo_id,
            repo_type="space",
            path_in_repo="",
        )
    finally:
        if copied_cfg and HF_CFG_STAGING.is_file():
            HF_CFG_STAGING.unlink(missing_ok=True)

    print(f"Uploaded {DEPLOY_DIR} to Space {repo_id}.")


if __name__ == "__main__":
    main()


Overwriting tourism_project/hosting/hosting.py


In [17]:
"""Push deployment bundle to the Hugging Face Space."""
import subprocess
import sys

subprocess.check_call([sys.executable, "tourism_project/hosting/hosting.py"])
print("Hosting upload complete.")


Uploaded /Users/narendrababu.sinappa/Desktop/mlops/tourism_project/deployment to Space snarendrababu41/wellness-tourism-streamlit.
Hosting upload complete.


### Observations — deployment, Space, and operations

- **Docker** + pinned deps match CI; **`0.0.0.0:7860`** and **`app_port: 7860`** let Hugging Face route traffic correctly.
- Set Space secrets **`HF_MODEL_REPO`** (and **`HF_TOKEN`** if the model repo is private).
- The **first Predict** may be slow while **`hf_hub_download`** runs; later clicks are faster.
- After changes, do a quick **manual smoke test** on the Space and know who fixes a failed deploy.


# MLOps Pipeline with Github Actions Workflow


**Note:**



1. Add `HF_TOKEN` and `HF_USER` to **GitHub → Settings → Secrets and variables → Actions**.

2. The workflow file below is also committed at `.github/workflows/pipeline.yml`. It runs on every **push to `main`** and via **workflow_dispatch**.

3. MLflow uses a **local file store** in `train.py` (no long-running MLflow server required in CI).



In [18]:
%%writefile tourism_project/requirements.txt
# Dependencies for GitHub Actions and local runs of tourism MLOps scripts.
huggingface_hub==0.32.6
datasets==3.6.0
pandas==2.2.2
scikit-learn==1.6.0
xgboost==2.1.4
mlflow==3.0.1
joblib==1.5.1


Overwriting tourism_project/requirements.txt


In [19]:
"""Ensure GitHub Actions workflow directory exists before writing pipeline.yml."""
import os

os.makedirs(".github/workflows", exist_ok=True)
print("Ready for .github/workflows/pipeline.yml")


Ready for .github/workflows/pipeline.yml


In [20]:
%%writefile .github/workflows/pipeline.yml
# End-to-end MLOps workflow for the Wellness Tourism purchase prediction project.
# Triggers on every push to main (CI/CD) and can be run manually.
#
# Required GitHub repository secrets:
#   HF_TOKEN   — Hugging Face token (write)
#   HF_USER    — Hugging Face username (used to build repo ids)
#
# Optional: set HF_DATASET_REPO, HF_MODEL_REPO, HF_SPACE_REPO as secrets if you
# use non-default repository names.

name: Wellness Tourism MLOps Pipeline

on:
  push:
    branches:
      - main
  workflow_dispatch:

jobs:
  register-dataset:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt

      - name: Ensure raw CSV in data folder
        run: |
          mkdir -p tourism_project/data
          if [ -f tourism.csv ]; then cp tourism.csv tourism_project/data/; fi
          if [ ! -f tourism_project/data/tourism.csv ]; then
            echo "tourism_project/data/tourism.csv missing; add tourism.csv at repo root or under tourism_project/data/"
            exit 1
          fi

      - name: Upload dataset to Hugging Face Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_USER: ${{ secrets.HF_USER }}
        run: python tourism_project/model_building/data_register.py

  data-prep:
    needs: register-dataset
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt

      - name: Run data preparation
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_USER: ${{ secrets.HF_USER }}
        run: python tourism_project/model_building/prep.py

  model-training:
    needs: data-prep
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt

      - name: Train, tune, log MLflow, push model to Hub
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_USER: ${{ secrets.HF_USER }}
        run: python tourism_project/model_building/train.py

  deploy-hosting:
    needs: model-training
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.10"

      - name: Install dependencies
        run: pip install -r tourism_project/requirements.txt

      - name: Push deployment files to Hugging Face Space
        env:
          HF_TOKEN: ${{ secrets.HF_TOKEN }}
          HF_USER: ${{ secrets.HF_USER }}
        run: python tourism_project/hosting/hosting.py


Overwriting .github/workflows/pipeline.yml


### Observations — GitHub Actions & end-to-end governance

- **`HF_TOKEN`** / **`HF_USER`** live in GitHub **secrets**, not in the notebook or repo.
- **`needs:`** chains **register → prep → train → deploy** so a bad step does not skip ahead.
- Use **pull requests** on `main` so broken training does not auto-publish to the Space.
- Later you can **split workflows** (e.g. deploy only when `deployment/` changes) to save time.


## Github Authentication and Push Files


In [21]:
# Local / Colab: configure git (uncomment and edit), then clone and copy project.
# !git config --global user.email "you@example.com"
# !git config --global user.name "Your Name"
# !git clone https://github.com/YOUR_USER/YOUR_REPO.git
# # Copy tourism_project and .github into the clone, then commit and push.


# Output Evaluation


- **GitHub:** repository link, screenshot of folder structure and a **green** workflow run.

- **Hugging Face Space:** public Streamlit app link and screenshot.



In [23]:
GITHUB_REPO_URL = "https://github.com/Naren2055/mlops/"
HF_SPACE_URL = "https://huggingface.co/spaces/snarendrababu41/wellness-tourism-streamlit"
HF_DATASET_URL = "https://huggingface.co/datasets/snarendrababu41/wellness-tourism-purchase"
HF_MODEL_URL = "https://huggingface.co/snarendrababu41/wellness-tourism-xgboost-model"

print("GitHub:", GITHUB_REPO_URL)
print("HF Space (Streamlit):", HF_SPACE_URL)
print("HF Dataset:", HF_DATASET_URL)
print("HF Model:", HF_MODEL_URL)
print("GitHub pipeline screenshot:", "https://huggingface.co/datasets/snarendrababu41/wellness-tourism-purchase/blob/main/Screenshot%202026-03-29%20at%205.45.50%E2%80%AFPM.png")

GitHub: https://github.com/Naren2055/mlops/
HF Space (Streamlit): https://huggingface.co/spaces/snarendrababu41/wellness-tourism-streamlit
HF Dataset: https://huggingface.co/datasets/snarendrababu41/wellness-tourism-purchase
HF Model: https://huggingface.co/snarendrababu41/wellness-tourism-xgboost-model
GitHub pipeline screenshot: https://huggingface.co/datasets/snarendrababu41/wellness-tourism-purchase/blob/main/Screenshot%202026-03-29%20at%205.45.50%E2%80%AFPM.png


## **Observations & insights**

- **Data:** Drop `CustomerID` to avoid leakage; impute consistently; normalise labels (e.g. `Fe Male` → `Female`) so training matches the app.
- **Model:** XGBoost with imbalance handling; grid search with **MLflow**; best weights pushed to the **Model Hub** for the Space to download.
- **MLOps:** Versioned data on the Hub, **Docker** Streamlit at `src/streamlit_app.py`, **GitHub Actions** on `main` for a repeatable path from CSV to UI.
- **Business:** Use scores to **prioritise outreach**; retrain when campaigns or data shift; revisit **privacy** and **fairness** if the use case expands.
- **Training vs serving:** New CRM values must be mapped before they appear in Streamlit, or scores can silently degrade (**training–serving skew**).
- **Calibration:** Rankings from the model are useful for sorting leads; treating probabilities as exact “% chance to buy” may need a **calibration** pass if finance uses them in forecasts.
- **Time:** Rows ordered by date would ideally use a **time-based validation** split later; the current random stratified split assumes rows are exchangeable.
- **Operations:** The Space **cold-starts** on first use (model download); **free-tier** limits and sleep are fine for demos but not for strict SLAs.

**Recommendations**

- Add a one-page **sales playbook**: when to trust the score, when to override, and how to log feedback for the next training round.
- Set **HF_MODEL_REPO** (and **HF_TOKEN** if private) on the Space; rotate **HF_TOKEN** in GitHub and Hub if it was ever pasted into a notebook output.
- Run **Kernel → Restart & Run All** before submission so errors and stale `HF_USER` values do not confuse reviewers.
- Plan a **calendar retrain** (e.g. quarterly) or trigger one when **conversion rate** or **feature distributions** shift materially.
- Keep **dataset / model / Space** repos aligned under the same **Hub username** as `HF_USER` and document the three URLs in the rubric screenshots.
